# 🎨 실시간 AI 비주얼 — 준비된 피아노 음악으로 시연하기

이 노트북은 **미리 준비된 피아노 음악**을 재생하면서, AI가 음악에 반응하는 이미지를 빠르게 생성하는 데모입니다.
수업에서는 업로드보다 먼저 아래의 예시 피아노 음원을 사용해 안정적으로 시연합니다.

**도구**: SD-Turbo (초경량 1-step 이미지 생성) + TAESD (초소형 디코더)

- 프레임당 약 0.2초 — **초당 3~5장** 생성
- 먼저 준비된 피아노 예시를 선택하고, 필요하면 나중에 자신의 파일로 바꿔볼 수 있습니다.

▶ 먼저 아래 셀을 실행해주세요. 설치에 2~3분 정도 걸립니다.

⚠️ **GPU 설정 확인**: 상단 메뉴에서 **런타임 → 런타임 유형 변경 → T4 GPU** 를 선택해주세요.
GPU가 설정되지 않으면 실행이 매우 느리거나 에러가 납니다.

In [ ]:
!pip install -q diffusers transformers accelerate librosa soundfile matplotlib Pillow

import warnings
warnings.filterwarnings('ignore')

print('✅ 설치 완료!')

---
## 1. 준비된 피아노 예시 선택

수업에서는 먼저 **이미 준비된 피아노 음원**으로 데모를 진행합니다.
아래 셀을 실행하면 세 가지 예시를 다운로드하고, 그중 하나를 바로 선택할 수 있습니다.

In [ ]:
import os
import urllib.request
import IPython.display as ipd

REPO_URL = 'https://raw.githubusercontent.com/joonhyungbae/pianokit/main/assets'

examples = {
    'piano_chopin.wav': '쇼팽 스타일 피아노 (서정적, 데모용 기본값)',
    'piano_jazz.wav': '재즈 피아노 즉흥',
    'piano_simple.wav': '짧고 단순한 피아노 프레이즈',
}

print('🎹 사용 가능한 준비된 피아노 예시')
for fname, desc in examples.items():
    if not os.path.exists(fname):
        try:
            urllib.request.urlretrieve(f'{REPO_URL}/{fname}', fname)
            print(f'  ✅ {fname} 다운로드 완료')
        except Exception:
            print(f'  ⚠️ {fname} 다운로드 실패')
    print(f'  - {fname}: {desc}')

# 수업용 기본 선택
input_audio = 'piano_chopin.wav'
# 다른 예시를 쓰고 싶다면 위 파일명 중 하나로 바꿔보세요.

print(f'\n🎵 현재 선택된 피아노 음원: {input_audio}')
if os.path.exists(input_audio):
    ipd.display(ipd.Audio(input_audio))
else:
    print('⚠️ 예시 파일이 없습니다. 아래 업로드 셀을 사용하세요.')

---
## 2. 오디오 분석

선택한 피아노 음원의 **음량(RMS)** 과 **비트**를 먼저 분석합니다.
이 값들이 뒤의 AI 비주얼 변화 강도와 장면 전환 타이밍에 사용됩니다.

▶ 아래 셀을 실행하세요.

In [ ]:
import librosa
import numpy as np

y, sr = librosa.load(input_audio, sr=22050)
duration = len(y) / sr

# RMS (음량)
rms = librosa.feature.rms(y=y)[0]
rms_norm = (rms - rms.min()) / (rms.max() - rms.min() + 1e-8)
rms_times = librosa.times_like(rms, sr=sr)

# Beat
tempo, beats = librosa.beat.beat_track(y=y, sr=sr)
tempo = float(tempo) if hasattr(tempo, '__len__') else float(tempo)
beat_times = librosa.frames_to_time(beats, sr=sr)

print(f'🎵 오디오: {duration:.1f}초, 템포 {tempo:.0f} BPM, {len(beat_times)}개 비트')

---
## 3. AI 모델 로드

이미지 생성 AI를 불러옵니다. 빠른 속도로 이미지를 만들어주는 모델입니다.

⏰ 모델 다운로드에 1~2분 걸립니다. ▶ 아래 셀을 실행하세요.

In [ ]:
import torch
from diffusers import AutoPipelineForText2Image, AutoPipelineForImage2Image, AutoencoderTiny

print('🔄 SD-Turbo + TAESD 모델을 불러오는 중...')

# SD-Turbo: 860M params, ~3.4GB 다운로드
model_id = 'stabilityai/sd-turbo'
# 더 높은 품질을 원하면 아래 모델로 교체 (6.5GB 다운로드, 프레임당 ~800ms):
# model_id = 'stabilityai/sdxl-turbo'

pipe_txt = AutoPipelineForText2Image.from_pretrained(
    model_id, torch_dtype=torch.float16, variant='fp16'
)

# TAESD: 초소형 디코더로 교체 (49M → 1.2M params, 디코딩 ~40배 빠름)
pipe_txt.vae = AutoencoderTiny.from_pretrained(
    'madebyollin/taesd', torch_dtype=torch.float16
).to('cuda')

pipe_txt.to('cuda')

# img2img — 같은 컴포넌트 공유 (추가 VRAM 없음)
pipe_img = AutoPipelineForImage2Image.from_pipe(pipe_txt)

print('✅ 모델 로딩 완료! (VRAM ~3.5GB)')

---
## 4. 프롬프트 설정

피아노 음악이 어떤 장면으로 보이길 원하는지 영어로 적어주세요.
수업에서는 먼저 준비된 예시 프롬프트로 빠르게 시연한 뒤, 한두 개만 바꿔보는 방식이 안정적입니다.

| 프롬프트 | 분위기 |
|---------|-------|
| `abstract flowing shapes, dark blue and gold` | 서정적 추상 금빛 |
| `cinematic piano stage, drifting particles, warm spotlight` | 무대 조명 느낌 |
| `ink wash landscape responding to piano resonance` | 수묵화 같은 질감 |

In [ ]:
prompt = "abstract flowing shapes, dark blue and gold"  # ← 여기를 수정하세요

print(f'🎨 프롬프트: "{prompt}"')

---
## 5. 실시간 생성

아래 셀을 실행하면 선택한 **피아노 음악**을 재생하면서 AI가 이미지를 계속 생성합니다.
음량이 커지면 이미지 변화가 커지고, 비트마다 장면이 조금씩 전환됩니다.

⏰ 수업 시연에서는 보통 15~30초 정도의 예시 음원으로 보여주는 것이 가장 안정적입니다.

In [ ]:
import time
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, Audio
from PIL import Image
import numpy as np

duration_sec = min(duration, 30)  # 최대 30초
fps_target = 4  # 목표 FPS

total_frames = int(duration_sec * fps_target)
print(f'🎬 {duration_sec:.0f}초, 목표 {fps_target}fps → 약 {total_frames}프레임')

# 비트 프레임 변환
beat_frame_set = set([int(bt * fps_target) for bt in beat_times if bt < duration_sec])

# 오디오를 별도 Output에 배치 (clear_output에 영향 안 받음)
audio_out = widgets.Output()
with audio_out:
    display(Audio(input_audio, autoplay=True))
display(audio_out)

# 이미지 표시용 별도 Output
image_out = widgets.Output()
display(image_out)

# 첫 프레임 생성
seed = 42
gen = torch.Generator('cuda').manual_seed(seed)

prev_image = pipe_txt(
    prompt=prompt,
    num_inference_steps=1,
    guidance_scale=0.0,
    generator=gen,
    width=512, height=512
).images[0]

start_time = time.time()
frame_times = []

for frame_idx in range(1, total_frames):
    t = frame_idx / fps_target

    rms_idx = min(int(t / duration * len(rms_norm)), len(rms_norm) - 1)
    current_rms = rms_norm[rms_idx]

    strength = 0.3 + current_rms * 0.5

    if frame_idx in beat_frame_set:
        seed += 1

    gen = torch.Generator('cuda').manual_seed(seed)

    frame_start = time.time()

    new_image = pipe_img(
        prompt=prompt,
        image=prev_image,
        strength=min(strength, 0.85),
        guidance_scale=0.0,
        num_inference_steps=1,
        generator=gen,
    ).images[0]

    frame_ms = (time.time() - frame_start) * 1000
    frame_times.append(frame_ms)

    # 이미지만 갱신 (오디오는 유지)
    with image_out:
        clear_output(wait=True)
        fig, ax = plt.subplots(1, 1, figsize=(7, 7))
        ax.imshow(new_image)
        ax.axis('off')
        ax.set_title(f'🎨 "{prompt[:35]}..."', fontsize=11)
        ax.set_xlabel(f't={t:.1f}s | {frame_ms:.0f}ms/frame | RMS={current_rms:.2f}', fontsize=9)
        plt.tight_layout()
        plt.show()
        plt.close(fig)

    prev_image = new_image

# 결과 요약
avg_ms = np.mean(frame_times)
print(f'\n✅ 완료! {len(frame_times)}프레임 생성')
print(f'⚡ 평균 {avg_ms:.0f}ms/프레임 ({1000/avg_ms:.1f} FPS)')

---
## 6. (선택) 피아노 예시와 프롬프트를 바꿔 다시 보기

- 먼저 1번 셀에서 다른 피아노 예시를 고르거나,
- 4번 프롬프트를 바꾼 뒤,
- 5번 셀을 다시 실행해보세요.

같은 방식의 생성이라도 **피아노 음악의 질감**과 **프롬프트의 방향**에 따라 전혀 다른 비주얼이 나옵니다.

💡 추천 실험:
- `piano_chopin.wav` + `cinematic piano stage, drifting particles, warm spotlight`
- `piano_jazz.wav` + `neon reflections, smoky club atmosphere, fluid motion`
- `piano_simple.wav` + `minimal geometric lines, white background, black ink`